# Data Preprocessing

## EDA

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv(r"D:\SIC\Projects\Cars price prediction\data\raw\used_cars_data.csv")

In [3]:
df.head()

,S.No.,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,New_Price,Price
0,0,Maruti Wagon R LXI CNG,Mumbai,2010,72000,CNG,Manual,First,26.6 km/kg,998 CC,58.16 bhp,5.0,NaN,1.75
1,1,Hyundai Creta 1.6 CRDi SX Option,Pune,2015,41000,Diesel,Manual,First,19.67 kmpl,1582 CC,126.2 bhp,5.0,NaN,12.50
2,2,Honda Jazz V,Chennai,2011,46000,Petrol,Manual,First,18.2 kmpl,1199 CC,88.7 bhp,5.0,8.61 Lakh,4.50
3,3,Maruti Ertiga VDI,Chennai,2012,87000,Diesel,Manual,First,20.77 kmpl,1248 CC,88.76 bhp,7.0,NaN,6.00
4,4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,2013,40670,Diesel,Automatic,Second,15.2 kmpl,1968 CC,140.8 bhp,5.0,NaN,17.74


In [4]:
df.shape

(7253, 14)

In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 7253 entries, 0 to 7252
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   S.No.              7253 non-null   int64  
 1   Name               7253 non-null   str    
 2   Location           7253 non-null   str    
 3   Year               7253 non-null   int64  
 4   Kilometers_Driven  7253 non-null   int64  
 5   Fuel_Type          7253 non-null   str    
 6   Transmission       7253 non-null   str    
 7   Owner_Type         7253 non-null   str    
 8   Mileage            7251 non-null   str    
 9   Engine             7207 non-null   str    
 10  Power              7207 non-null   str    
 11  Seats              7200 non-null   float64
 12  New_Price          1006 non-null   str    
 13  Price              6019 non-null   float64
dtypes: float64(2), int64(3), str(9)
memory usage: 1.3 MB


In [6]:
df.isnull().sum()

S.No.                   0
Name                    0
Location                0
Year                    0
Kilometers_Driven       0
Fuel_Type               0
Transmission            0
Owner_Type              0
Mileage                 2
Engine                 46
Power                  46
Seats                  53
New_Price            6247
Price                1234
dtype: int64

In [7]:
print(df.duplicated().sum())

0


In [8]:
df["Seats"].value_counts()

Seats
5.0     6047
7.0      796
8.0      170
4.0      119
6.0       38
2.0       18
10.0       8
9.0        3
0.0        1
Name: count, dtype: int64

# Cleaning

##### Dropping S.No column because its useless and New_Price because it has many null values

In [9]:
df = df.drop(columns=['S.No.', 'New_Price'])

print(df.shape)

(7253, 12)


##### Dropping null rows in price because its our target, so we cannot impute its values

In [10]:
df = df.dropna(subset=['Price'])

print(df.isnull().sum())

Name                  0
Location              0
Year                  0
Kilometers_Driven     0
Fuel_Type             0
Transmission          0
Owner_Type            0
Mileage               2
Engine               36
Power                36
Seats                42
Price                 0
dtype: int64


In [11]:
print(df.shape)

(6019, 12)


## Mileage cleaning and normalizing

In [12]:
df['Mileage'].str.split(' ').str[1].unique()

array(['km/kg', 'kmpl', nan], dtype=object)

### now we will remove the text from the Mileage and save the text in new column to save the relation between 'km/kg', 'kmpl'

In [13]:
df['Mileage_Unit'] = df['Mileage'].str.split(' ').str[1]
df['Mileage'] = df['Mileage'].str.split(' ').str[0].astype(float)

In [14]:
print(df['Mileage'].dtype)


float64


In [15]:
df[['Mileage', 'Mileage_Unit']].head(10)

,Mileage,Mileage_Unit
0,26.60,km/kg
1,19.67,kmpl
2,18.20,kmpl
3,20.77,kmpl
4,15.20,kmpl
5,21.10,km/kg
6,23.08,kmpl
7,11.36,kmpl
8,20.54,kmpl
9,22.30,kmpl


In [16]:
df.isnull().sum()

Name                  0
Location              0
Year                  0
Kilometers_Driven     0
Fuel_Type             0
Transmission          0
Owner_Type            0
Mileage               2
Engine               36
Power                36
Seats                42
Price                 0
Mileage_Unit          2
dtype: int64

### Dropping Electric cars
Electric cars have no meaningful Mileage (the concept doesn't apply), which is why their Mileage/Mileage_Unit were NaN from the source data. We drop them here — this is a row-level rule (based only on Fuel_Type), so it's safe to do before the train/test split. This also resolves the Mileage/Mileage_Unit nulls automatically.

In [17]:
df = df[df['Fuel_Type'] != 'Electric']

print(df.isnull().sum())

Name                  0
Location              0
Year                  0
Kilometers_Driven     0
Fuel_Type             0
Transmission          0
Owner_Type            0
Mileage               0
Engine               36
Power                36
Seats                42
Price                 0
Mileage_Unit          0
dtype: int64


## Engine cleaning and normalizing

In [18]:
df['Engine'].str.split(' ').str[1].unique()

array(['CC', nan], dtype=object)

### we dont need to add new column t save the text because all values are CC

In [19]:
df['Engine'] = df['Engine'].str.split(' ').str[0].astype(float)

In [20]:
df['Engine'].dtype

dtype('float64')

In [21]:
df[["Engine"]].head(5)

,Engine
0,998.0
1,1582.0
2,1199.0
3,1248.0
4,1968.0


## Power cleaning and normalizing

In [22]:
df['Power'].str.split(' ').str[1].unique()

array(['bhp', nan], dtype=object)

### same as Engine we dont need to save the text value "bhp"

### but Power has "null" values as text not just NaN, so we convert it first

In [23]:
df[df['Power'].str.contains('null', na=False)]

,Name,Location,Year,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Mileage_Unit
76,Ford Fiesta 1.4 SXi TDCi,Jaipur,2008,111111,Diesel,Manual,First,17.80,1399.0,null bhp,5.0,2.00,kmpl
79,Hyundai Santro Xing XL,Hyderabad,2005,87591,Petrol,Manual,First,0.00,1086.0,null bhp,5.0,1.30,kmpl
89,Hyundai Santro Xing XO,Hyderabad,2007,73745,Petrol,Manual,First,17.00,1086.0,null bhp,5.0,2.10,kmpl
120,Hyundai Santro Xing XL eRLX Euro III,Mumbai,2005,102000,Petrol,Manual,Second,17.00,1086.0,null bhp,5.0,0.85,kmpl
143,Hyundai Santro Xing XO eRLX Euro II,Kochi,2008,80759,Petrol,Manual,Third,17.00,1086.0,null bhp,5.0,1.67,kmpl
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5873,Hyundai Santro Xing XO eRLX Euro II,Pune,2006,47200,Petrol,Manual,Second,17.00,1086.0,null bhp,5.0,1.20,kmpl
5893,Maruti Estilo LXI,Chennai,2008,51000,Petrol,Manual,Second,19.50,1061.0,null bhp,NaN,1.75,kmpl
5925,Skoda Laura Classic 1.8 TSI,Pune,2010,85000,Petrol,Manual,First,17.50,1798.0,null bhp,5.0,2.85,kmpl
5943,Mahindra Jeep MM 540 DP,Chennai,2002,75000,Diesel,Manual,First,0.00,2112.0,null bhp,6.0,1.70,kmpl


### checking on all columns

In [24]:
suspicious = ['null', 'na', 'n/a', 'none', 'nan']

for col in df.columns:
    if df[col].dtype == 'object':
        mask = df[col].astype(str).str.strip().str.lower().str.contains('|'.join(suspicious), na=False, regex=True)
        if mask.sum() > 0:
            print(col, '->', mask.sum(), 'suspicious values')

In [25]:
mask_name = df['Name'].astype(str).str.strip().str.lower().str.contains('|'.join(['null', 'na', 'n/a', 'none', 'nan']), na=False, regex=True)
print(df.loc[mask_name, 'Name'].unique()[:20])

print("------------")

mask_loc = df['Location'].astype(str).str.strip().str.lower().str.contains('|'.join(['null', 'na', 'n/a', 'none', 'nan']), na=False, regex=True)
print(df.loc[mask_loc, 'Location'].unique())

<ArrowStringArray>
[       'Renault Duster 85PS Diesel RxL Plus',
                      'Hyundai i20 1.2 Magna',
       'Renault Duster 110PS Diesel RxZ Pack',
                            'Tata Nano LX SE',
             'Hyundai i20 Magna Optional 1.2',
                           'Renault KWID RXT',
 'Ford Ecosport 1.5 DV5 MT Titanium Optional',
        'Hyundai Verna Transform SX VGT CRDi',
           'Hyundai Verna 1.6 SX CRDI (O) AT',
                         'Tata Nano Twist XT',
           'Hyundai Verna CRDi 1.6 SX Option',
                              'Tata Nano XTA',
          'Hyundai Verna VTVT 1.6 AT SX Plus',
                     'Hyundai EON Magna Plus',
                  'Maruti Swift VXI Optional',
                      'Hyundai i10 Magna 1.2',
           'Renault Lodgy 110PS RxZ 8 Seater',
                       'Hyundai Verna 1.6 SX',
                      'Hyundai i10 Magna 1.1',
        'Land Rover Range Rover 2.2L Dynamic']
Length: 20, dtype: str
------------
<Arro

### Name -> 596 suspicious values
### Location -> 494 suspicious values
### are false positive, thanks God
### so wel will focus only on:
### Power -> 143 suspicious values
### Mileage_Unit -> 2 suspicious values

In [26]:
mask_power = df['Power'].astype(str).str.strip().str.lower().str.contains(r'\bnull\b', na=False, regex=True)
print(mask_power.sum())
print(df.loc[mask_power, 'Power'].unique())

107
<ArrowStringArray>
['null bhp']
Length: 1, dtype: str


### positive false 107

#### now we replace the 107 "null" values in Power with nan

In [27]:
df['Power'] = df['Power'].replace('null bhp', np.nan)

In [28]:
df['Power'] = df['Power'].str.split(' ').str[0].astype(float)

In [29]:
print(df['Power'].dtype)

float64


In [30]:
print(df['Power'].isnull().sum())

143


##### 36+107=143
##### 36 -> real nulls
##### 107 "null" replaced with nan

In [31]:
df.isnull().sum()

Name                   0
Location               0
Year                   0
Kilometers_Driven      0
Fuel_Type              0
Transmission           0
Owner_Type             0
Mileage                0
Engine                36
Power                143
Seats                 42
Price                  0
Mileage_Unit           0
dtype: int64

## Seats cleaning 

In [32]:
df["Seats"].value_counts()

Seats
5.0     5012
7.0      674
8.0      134
4.0       99
6.0       31
2.0       16
10.0       5
9.0        3
0.0        1
Name: count, dtype: int64

In [33]:
df.loc[df['Seats'] == 0, 'Seats'] = np.nan

In [34]:
df['Seats'].value_counts(dropna=False)

Seats
5.0     5012
7.0      674
8.0      134
4.0       99
NaN       43
6.0       31
2.0       16
10.0       5
9.0        3
Name: count, dtype: int64

## Feature Engineering in Year

In [35]:
print(df['Year'].max())
print(df['Year'].min())

2019
1998


In [36]:
df["Year"].value_counts()

Year
2014    797
2015    744
2016    740
2013    649
2017    587
2012    580
2011    465
2010    342
2018    298
2009    198
2008    174
2007    125
2019    102
2006     78
2005     57
2004     31
2003     17
2002     15
2001      8
2000      4
1998      4
1999      2
Name: count, dtype: int64

#### we will use 2020 as a refrence point to calculate car_age

In [37]:
df["Car_Age"] = df["Year"].max() + 1 - df["Year"] 

##### we didnt use 2020 as it is, so anyone can use the same code if we addedd more samples with new Years, so this makes oue model more **Scalable**

In [38]:
df = df.drop(columns=["Year"])

In [39]:
print(df[['Car_Age']].describe())

           Car_Age
count  6017.000000
mean      6.641848
std       3.269967
min       1.000000
25%       4.000000
50%       6.000000
75%       9.000000
max      22.000000


In [40]:
df.head()

,Name,Location,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Mileage_Unit,Car_Age
0,Maruti Wagon R LXI CNG,Mumbai,72000,CNG,Manual,First,26.60,998.0,58.16,5.0,1.75,km/kg,10
1,Hyundai Creta 1.6 CRDi SX Option,Pune,41000,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,kmpl,5
2,Honda Jazz V,Chennai,46000,Petrol,Manual,First,18.20,1199.0,88.70,5.0,4.50,kmpl,9
3,Maruti Ertiga VDI,Chennai,87000,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,kmpl,8
4,Audi A4 New 2.0 TDI Multitronic,Coimbatore,40670,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,kmpl,7


## Name cleaning and normalizing

In [41]:
df["Name"].value_counts()

Name
Mahindra XUV500 W8 2WD                        49
Maruti Swift VDI                              45
Honda City 1.5 S MT                           34
Maruti Swift Dzire VDI                        34
Maruti Swift VDI BSIV                         31
                                              ..
Hyundai Elantra SX                             1
Maruti Wagon R Duo Lxi                         1
Volkswagen Polo IPL II 1.2 Petrol Highline     1
Tata Bolt Revotron XT                          1
Mahindra Xylo D4 BSIV                          1
Name: count, Length: 1874, dtype: int64

#### Name has too much variety (1876 unique values) to encode directly. We'll extract just the first word (the brand) that drastically reduces the variety while keeping meaningful information for the model.

In [42]:
df['Name'].nunique()

1874

##### toooooo muchhh variety

In [43]:
df['Brand'] = df['Name'].str.split(' ').str[0]

In [44]:
df['Brand'].nunique()

31

In [45]:
df['Brand'].value_counts()

Brand
Maruti           1211
Hyundai          1107
Honda             608
Toyota            410
Mercedes-Benz     318
Volkswagen        315
Ford              300
Mahindra          271
BMW               267
Audi              236
Tata              186
Skoda             173
Renault           145
Chevrolet         121
Nissan             91
Land               60
Jaguar             40
Fiat               28
Mitsubishi         27
Mini               26
Volvo              21
Porsche            18
Jeep               15
Datsun             13
Force               3
ISUZU               2
Smart               1
Ambassador          1
Isuzu               1
Bentley             1
Lamborghini         1
Name: count, dtype: int64

##### we can see that there is no Brand has name "Land", its "Land Rover" and ISUZU Isuzu                we will customize that:               

In [46]:
df['Brand'] = df['Brand'].str.title()

df['Brand'] = df['Brand'].replace('Land', 'Land Rover')

In [47]:
print(df['Brand'].nunique())
print(df['Brand'].value_counts())

30
Brand
Maruti           1211
Hyundai          1107
Honda             608
Toyota            410
Mercedes-Benz     318
Volkswagen        315
Ford              300
Mahindra          271
Bmw               267
Audi              236
Tata              186
Skoda             173
Renault           145
Chevrolet         121
Nissan             91
Land Rover         60
Jaguar             40
Fiat               28
Mitsubishi         27
Mini               26
Volvo              21
Porsche            18
Jeep               15
Datsun             13
Isuzu               3
Force               3
Smart               1
Ambassador          1
Bentley             1
Lamborghini         1
Name: count, dtype: int64


##### to return BMW as it is 

In [48]:
df['Brand'] = df['Brand'].replace('Bmw', 'BMW')

In [49]:
print(df['Brand'].nunique())
print(df['Brand'].value_counts())

30
Brand
Maruti           1211
Hyundai          1107
Honda             608
Toyota            410
Mercedes-Benz     318
Volkswagen        315
Ford              300
Mahindra          271
BMW               267
Audi              236
Tata              186
Skoda             173
Renault           145
Chevrolet         121
Nissan             91
Land Rover         60
Jaguar             40
Fiat               28
Mitsubishi         27
Mini               26
Volvo              21
Porsche            18
Jeep               15
Datsun             13
Isuzu               3
Force               3
Smart               1
Ambassador          1
Bentley             1
Lamborghini         1
Name: count, dtype: int64


In [50]:
df = df.drop(columns=['Name'])
print(df.columns.tolist())

['Location', 'Kilometers_Driven', 'Fuel_Type', 'Transmission', 'Owner_Type', 'Mileage', 'Engine', 'Power', 'Seats', 'Price', 'Mileage_Unit', 'Car_Age', 'Brand']


# Save stage 1 (cleaned, row-level only, no split/imputation/encoding yet)

In [51]:
df.to_csv('../data/processed/cars_stage1_cleaned.csv', index=False)

print(df.shape)
df.head()

(6017, 13)


,Location,Kilometers_Driven,Fuel_Type,Transmission,Owner_Type,Mileage,Engine,Power,Seats,Price,Mileage_Unit,Car_Age,Brand
0,Mumbai,72000,CNG,Manual,First,26.60,998.0,58.16,5.0,1.75,km/kg,10,Maruti
1,Pune,41000,Diesel,Manual,First,19.67,1582.0,126.20,5.0,12.50,kmpl,5,Hyundai
2,Chennai,46000,Petrol,Manual,First,18.20,1199.0,88.70,5.0,4.50,kmpl,9,Honda
3,Chennai,87000,Diesel,Manual,First,20.77,1248.0,88.76,7.0,6.00,kmpl,8,Maruti
4,Coimbatore,40670,Diesel,Automatic,Second,15.20,1968.0,140.80,5.0,17.74,kmpl,7,Audi


# Data Cleaning Summary (Stage 1 — 01_data_cleaning.ipynb)

- Loaded raw data (7,253 rows), checked missing values and dtypes
- Dropped `S.No.` (useless index) and `New_Price` (86% missing)
- Dropped rows with missing `Price` (target — can't impute it)
- Split unit text from numbers in `Mileage`, `Engine`, `Power`
- Found a hidden "null bhp" text value in `Power`, converted it to NaN before splitting
- Fixed `Seats` = 0 (invalid) → converted to NaN
- Dropped Electric cars (Mileage has no meaningful value for them) — this also resolved the `Mileage`/`Mileage_Unit` nulls
- Engineered `Car_Age` from `Year`, dropped `Year`
- Extracted `Brand` from `Name`, fixed casing/naming inconsistencies (Land Rover, BMW), dropped `Name`

Saved as `cars_stage1_cleaned.csv` — cleaned at the row level only, still has missing values (Engine, Power, Seats), not split into train/test yet.